<a href="https://colab.research.google.com/github/DawerSP/Inteligencia-Aritificial/blob/main/RegresionLog%C3%ADstica_cancer_cervical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# REGRESIÓN LOGÍSTICA - CÁNCER CERVICAL
# ==========================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score
)
import matplotlib.pyplot as plt
import seaborn as sns



In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00383/risk_factors_cervical_cancer.csv"
df = pd.read_csv(url)

print("Tamaño original:", df.shape)
display(df.head())

In [ ]:
df = df.replace("?", np.nan)
df = df.astype(float)

# Eliminar filas con demasiados valores nulos
df = df.dropna(thresh=10)   # al menos 10 valores no nulos
df = df.fillna(df.median()) # llenar faltantes con mediana

print("Tamaño después de limpieza:", df.shape)


In [ ]:
features = [
    "Age", "Number of sexual partners", "First sexual intercourse",
    "Num of pregnancies", "Smokes", "Hormonal Contraceptives",
    "STDs", "STDs:HPV"
]

target = "Biopsy"

X = df[features]
y = df[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train_scaled, y_train)


In [ ]:
y_pred = modelo.predict(X_test_scaled)
y_prob = modelo.predict_proba(X_test_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nMatriz de confusión:\n", confusion_matrix(y_test, y_pred))
print("\nReporte de clasificación:\n", classification_report(y_test, y_pred))

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(7,6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}", color="darkorange")
plt.plot([0,1],[0,1],"k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC - Cáncer Cervical (Regresión Logística)")
plt.legend()
plt.show()


In [ ]:
coef = pd.DataFrame({
    "Variable": features,
    "Coeficiente": modelo.coef_[0]
}).sort_values("Coeficiente", ascending=False)

print("\nCoeficientes del modelo (importancia):")
display(coef)